# Educational WebCrawler – Tower Edition (Run All)

**Zweck:** Selbstständig Unterrichtsarbeitsblätter, Lernpfade, Unterrichtsverlaufspläne und methodisch-didaktische Handreichungen aus kuratierten Bildungsquellen erfassen und lokal ablegen. **Ein Klick auf „Run All“ startet den gesamten Prozess** inklusive Paketprüfung, Crawl, zugelassenem Materialdownload, Textextraktion und Export.

- Standardmäßig ist **`MODUS = "NACHT"`** und **`START_CRAWL = True`** eingestellt.
- Die drei Modi **TEST / STANDARD / NACHT** sind in der nächsten Einstellungszelle kommentiert.
- Speicherziel: `outputs/runs/tower_nacht/` (inklusive `materials/`, `documents.jsonl`, `search_documents.jsonl`, `crawl.jsonl`, Berichten und `crawler_state.sqlite`).
- Unterbrechungen sind unkritisch: Beim nächsten Run All wird dieselbe Sitzung anhand SQLite **fortgesetzt**. Für einen komplett neuen Crawl `RUN_NAME` ändern.
- **300 MiB sind ein Zielwert für den bereinigten JSONL-Suchkorpus, keine Garantie.** Qualität und zulässige Inhalte haben Vorrang vor Datenmenge.
- Es werden **nur freigegebene Bildungsdomains** verfolgt, auch domainübergreifend, und robots.txt beachtet. Kein unbegrenzter Webcrawl.
- Dateien (PDF/DOCX/PPTX/ODT/TXT) werden bei **nachweisbarer oder für eine Quelle ausdrücklich hinterlegter offener Lizenz** gespeichert. PDFs ohne Lizenzhinweis auf der Quellseite werden optional im Arbeitsspeicher auf ihre eigenen Lizenzangaben überprüft; ohne Nachweis werden sie verworfen. Bei unklarer Lizenz bleibt der Materiallink in den Metadaten. Kein Umgehen von Logins, Paywalls oder Zugriffssperren.
- HTML-Seiten mit Aufgaben/Unterrichtsschritten/Methoden werden als Suchdokumente gespeichert; Navigationsseiten dienen nur zum Finden weiterer Inhalte.

**Hinweis:** JavaScript-renderte Inhalte und reine Scan-PDFs ohne Textschicht werden nicht vollständig ausgelesen; dafür wäre ein separater, quellengebundener Extraktor nötig.

In [ ]:
# ================================================================
# EINSTELLUNGEN: diese Zelle ist die einzige, die du ändern musst.
# ================================================================
# MODUS = "TEST"      ->  ca. 70 Seiten, 6 min, Tiefe 3 (Funktionskontrolle)
# MODUS = "STANDARD"  ->  ca. 15.000 Seiten, 3 h, Tiefe 8
# MODUS = "NACHT"     ->  bis zu 120.000 Seiten, 12 h, Tiefe 12
MODUS = "NACHT"         # Standard für den Tower: langer Sammellauf
START_CRAWL = True      # Wichtig: Run All crawlt tatsächlich! (False = nur Offline-Test)
RUN_NAME = "tower_nacht"  # gleicher Name = Lauf fortsetzen; neuer Name = neuer Korpus
TARGET_MIB = 300        # Ziel für documents.jsonl, nicht crawl.jsonl!

# Inhalte und Materialien
DOWNLOAD_MATERIALS = True          # versucht PDF, DOCX, PPTX, ODT, TXT herunterzuladen
ONLY_OPEN_LICENSE_MATERIALS = True # nur offen lizenzierte Dateien werden dauerhaft gespeichert
CHECK_LICENSE_IN_PDFS = True       # ungeklärte PDF kurz im RAM prüfen; ohne Lizenzhinweis verwerfen
SITEMAPS_AKTIV = True             # findet zusätzliche Bildungsunterseiten
QUELLEN_UEBERGREIFEND = True      # nur innerhalb der freigegebenen Hostliste
MIN_TEXT_CHARS = 350             # sehr kurze HTML-Texte nicht indexieren
MAX_MATERIAL_MIB = 20            # einzelne Dateien maximal 20 MiB
DOWNLOAD_DELAY_SECONDS = 1.2     # minimale Pause pro Domain; robots crawl-delay gilt zusätzlich

# Das hier sollte für den Tower zunächst unverändert bleiben.
PROFILE = {
    "TEST": dict(max_pages=70, runtime_minutes=6,  max_depth=3,  workers=3, max_per_host=30, max_queue=800),
    "STANDARD": dict(max_pages=15000, runtime_minutes=180, max_depth=8, workers=6, max_per_host=4000, max_queue=55000),
    "NACHT": dict(max_pages=120000, runtime_minutes=720, max_depth=12, workers=8, max_per_host=30000, max_queue=350000),
}[MODUS]
assert MODUS in {"TEST", "STANDARD", "NACHT"}
assert RUN_NAME.replace("_", "").replace("-", "").isalnum(), "RUN_NAME: nur Buchstaben/Ziffern/Unterstriche/Minus"
print(f"Modus: {MODUS} | Run All aktiv: {START_CRAWL} | Ziel: {TARGET_MIB} MiB | Profil: {PROFILE}", flush=True)

## 1. Abhängigkeiten automatisch installieren
Diese Zelle verwendet den **aktiven Jupyter-Kernel** (keine feste Windows-Python-Version). Beim ersten Mal benötigt sie Internetzugang. Bei erneuten Durchläufen werden bereits installierte Bibliotheken nicht neu installiert.

In [ ]:
import sys, subprocess, importlib.util
REQUIRED = {
    "requests": "requests>=2.31,<3",
    "bs4": "beautifulsoup4>=4.12,<5",
    "lxml": "lxml>=5,<7",
    "pypdf": "pypdf>=5,<7",
    "docx": "python-docx>=1,<2",
    "pptx": "python-pptx>=1,<2",
}
missing = [pkg for mod,pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installiere fehlende Pakete:", missing, flush=True)
    subprocess.check_call([sys.executable,"-m","pip","install",*missing])
else:
    print("Alle Abhängigkeiten vorhanden.", flush=True)
print("Aktiver Python-Kernel:",sys.executable, flush=True)

## 2. Bildungsquellen / Seed-Liste
Die bestehenden 18 Quellengruppen sind erhalten. Ergänzt wurden konkrete Arbeitsblatt- und Downloadübersichten sowie weitere OER-Portale. Links auf andere freigegebene Quellen dürfen gefolgt werden, **nicht** jedoch beliebige Links ins gesamte Web. Fehlerhafte Seeds werden protokolliert und blockieren andere Quellen nicht.

In [ ]:
SOURCES = {'zum_unterrichten': {'hosts': ['unterrichten.zum.de'], 'seeds': ['https://unterrichten.zum.de/wiki/Unterrichtsideen', 'https://unterrichten.zum.de/wiki/Methode', 'https://unterrichten.zum.de/wiki/Kategorie:Lernpfad', 'https://unterrichten.zum.de/wiki/Kategorie:Unterrichtsidee', 'https://unterrichten.zum.de/wiki/Kategorie:Methode', 'https://unterrichten.zum.de/wiki/Informatik', 'https://unterrichten.zum.de/wiki/Geschichte', 'https://unterrichten.zum.de/wiki/Deutsch', 'https://unterrichten.zum.de/wiki/Mathematik', 'https://unterrichten.zum.de/wiki/Ethik', 'https://unterrichten.zum.de/wiki/Geographie', 'https://unterrichten.zum.de/wiki/Lernpfad', 'https://unterrichten.zum.de/wiki/Industrielle_Revolution', 'https://unterrichten.zum.de/wiki/Einführung_in_die_Automatentheorie', 'https://unterrichten.zum.de/wiki/Kategorie:Informatik', 'https://unterrichten.zum.de/wiki/Kategorie:Arbeitsblatt']}, 'zum_deutsch_lernen': {'hosts': ['deutsch-lernen.zum.de'], 'seeds': ['https://deutsch-lernen.zum.de/wiki/Unterrichtsmethoden', 'https://deutsch-lernen.zum.de/wiki/Methoden-_und_Spielekiste', 'https://deutsch-lernen.zum.de/wiki/Kategorie:Methoden', 'https://deutsch-lernen.zum.de/wiki/Kategorie:Methoden_und_Spiele', 'https://deutsch-lernen.zum.de/wiki/Übung', 'https://deutsch-lernen.zum.de/wiki/Gruppenpuzzle', 'https://deutsch-lernen.zum.de/wiki/Stationenlernen']}, 'serlo': {'hosts': ['de.serlo.org'], 'seeds': ['https://de.serlo.org/', 'https://de.serlo.org/mathe', 'https://de.serlo.org/informatik', 'https://de.serlo.org/biologie', 'https://de.serlo.org/chemie', 'https://de.serlo.org/informatik', 'https://de.serlo.org/biologie', 'https://de.serlo.org/physik']}, 'segu_geschichte': {'hosts': ['segu-geschichte.de', 'www.segu-geschichte.de'], 'seeds': ['https://segu-geschichte.de/', 'https://segu-geschichte.de/arbeitsblaetter/', 'https://segu-geschichte.de/lernbausteine/']}, 'edulabs': {'hosts': ['edulabs.de', 'www.edulabs.de'], 'seeds': ['https://edulabs.de/oer/', 'https://edulabs.de/blog/']}, 'digitallearninglab': {'hosts': ['digitallearninglab.de', 'www.digitallearninglab.de'], 'seeds': ['https://digitallearninglab.de/unterrichtsbausteine', 'https://digitallearninglab.de/unterrichtsbausteine/programmieren-mit-logo', 'https://digitallearninglab.de/unterrichtsbausteine/gib-den-stick-weiter-freies-schreiben-und-fortsetz']}, 'planet_schule': {'hosts': ['planet-schule.de', 'www.planet-schule.de'], 'seeds': ['https://www.planet-schule.de/index.html', 'https://www.planet-schule.de/schwerpunkt/marie-meets-marx/arbeit-unterricht-100.html']}, 'bpb': {'hosts': ['bpb.de', 'www.bpb.de'], 'seeds': ['https://www.bpb.de/lernen/angebote/', 'https://www.bpb.de/shop/materialien/themenblaetter/', 'https://www.bpb.de/shop/materialien/themenblaetter/311145/die-themenblaetter-als-oer/', 'https://www.bpb.de/lernen/', 'https://www.bpb.de/shop/materialien/themenblaetter/']}, 'lehrerfortbildung_bw': {'hosts': ['lehrerfortbildung-bw.de', 'www.lehrerfortbildung-bw.de'], 'seeds': ['https://lehrerfortbildung-bw.de/u_matnatech/', 'https://lehrerfortbildung-bw.de/u_matnatech/imp/gym/bp2016/fb1/', 'https://lehrerfortbildung-bw.de/u_matnatech/imp/gym/bp2016/fb2/', 'https://lehrerfortbildung-bw.de/u_matnatech/imp/gym/bp2016/fb2/z_download/']}, 'medien_in_die_schule': {'hosts': ['www.medien-in-die-schule.de', 'medien-in-die-schule.de'], 'seeds': ['https://www.medien-in-die-schule.de/unterrichtseinheiten/', 'https://www.medien-in-die-schule.de/unterrichtseinheiten/jugendliche-online/', 'https://www.medien-in-die-schule.de/unterrichtseinheiten/jugendliche-online/modul-1-persoenliche-daten/', 'https://www.medien-in-die-schule.de/unterrichtseinheiten/jugendliche-online/uebersicht-aller-material-und-arbeitsblaetter/', 'https://www.medien-in-die-schule.de/unterrichtseinheiten/meinung-im-netz-gestalten/', 'https://www.medien-in-die-schule.de/unterrichtseinheiten/meinung-im-netz-gestalten/uebersicht-aller-material-und-arbeitsblaetter/', 'https://www.medien-in-die-schule.de/downloads/', 'https://www.medien-in-die-schule.de/downloads/']}, 'klicksafe': {'hosts': ['www.klicksafe.de', 'klicksafe.de'], 'seeds': ['https://www.klicksafe.de/materialien', 'https://www.klicksafe.de/materialien/wo-ist-klaus']}, 'bildungsserver_berlin_brandenburg': {'hosts': ['bildungsserver.berlin-brandenburg.de'], 'seeds': ['https://bildungsserver.berlin-brandenburg.de/unterrichtsmaterialien', 'https://bildungsserver.berlin-brandenburg.de/17067', 'https://bildungsserver.berlin-brandenburg.de/sonderpaedagogik/medienbildung-im-sonderpaedagogischen-foerderschwerpunkt-geistige-entwicklung']}, 'bildungsserver_rlp': {'hosts': ['bildung.rlp.de'], 'seeds': ['https://bildung.rlp.de/unterricht', 'https://bildung.rlp.de/unterricht/faecher', 'https://bildung.rlp.de/verbraucherbildung/richtlinien-und-materialien']}, 'bildungsserver_hessen': {'hosts': ['lernarchiv.bildung.hessen.de'], 'seeds': ['https://lernarchiv.bildung.hessen.de/', 'https://lernarchiv.bildung.hessen.de/grundschule']}, 'deutscher_bildungsserver': {'hosts': ['www.bildungsserver.de', 'bildungsserver.de'], 'seeds': ['https://www.bildungsserver.de/schule/unterricht-634-de.html', 'https://www.bildungsserver.de/schule/grundschule-unterrichtsmaterial-1661-de.html']}, 'kinofenster': {'hosts': ['www.kinofenster.de', 'kinofenster.de'], 'seeds': ['https://www.kinofenster.de/unterrichten/', 'https://www.kinofenster.de/unterrichten/unterrichtsmaterial', 'https://www.kinofenster.de/unterricht/']}, 'hanisauland': {'hosts': ['www.hanisauland.de', 'hanisauland.de'], 'seeds': ['https://www.hanisauland.de/lehrer-innen/', 'https://www.hanisauland.de/lehrer-innen/unterrichtsmaterial/lehrer-innen-materialien-bestellen']}, 'bildung_lsa': {'hosts': ['www.bildung-lsa.de', 'bildung-lsa.de'], 'seeds': ['https://www.bildung-lsa.de/informationsportal/schule/schulentwicklung/digitalassistenz/freie_bildungsmaterialien.htm']}, 'environment_education': {'hosts': ['www.umwelt-im-unterricht.de', 'umwelt-im-unterricht.de'], 'seeds': ['https://www.umwelt-im-unterricht.de/', 'https://www.umwelt-im-unterricht.de/unterrichtsvorschlaege/', 'https://www.umwelt-im-unterricht.de/materialien/']}, 'siemens_open_mint': {'hosts': ['medienportal.siemens-stiftung.org'], 'seeds': ['https://medienportal.siemens-stiftung.org/de/home']}}

SEEDS = list(dict.fromkeys(u for cfg in SOURCES.values() for u in cfg["seeds"]))
HOST_TO_SOURCE = {h.lower():k for k,v in SOURCES.items() for h in v["hosts"]}
ALLOWED_HOSTS = set(HOST_TO_SOURCE)
# Quellen mit explizit nachgewiesener durchgängiger offener Materiallizenz:
SITE_LICENSE = {"segu-geschichte.de":"CC BY-SA 4.0", "www.segu-geschichte.de":"CC BY-SA 4.0"}
print(f"{len(SEEDS)} Seed-URLs, {len(SOURCES)} Quellengruppen, {len(ALLOWED_HOSTS)} freigegebene Hostnamen", flush=True)
for k,cfg in SOURCES.items():
    print(f" - {k}: {len(cfg['seeds'])} Seeds")

## 3. Crawl-Engine
Die Engine ist in diesem Notebook enthalten – **keine zusätzlichen Projektdateien erforderlich**. Sie verwendet SQLite als wiederaufnehmbare URL-Warteschlange, kontrolliert robots.txt pro Host, lädt Materialdateien herunter und erzeugt bereinigte Unterrichtsdokumente. Fortschritt erscheint während des Crawls direkt unterhalb der Startzelle.

In [ ]:
import concurrent.futures, gzip, hashlib, html, io, json, re, sqlite3, threading, time
import urllib.robotparser, zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit, urljoin, parse_qsl, urlencode
from xml.etree import ElementTree as ET
import requests
from bs4 import BeautifulSoup

USER_AGENT = 'EducationalIRCrawler/1.0 (academic educational materials research; respects robots.txt)'
TRACKING = {'fbclid','gclid','_ga','_gl','msclkid','ref_src'}
SUFFIXES = ('.pdf','.docx','.pptx','.odt','.odp','.txt')
NEG_URL = re.compile(r'(^|[/_?&=-])(datenschutz|impressum|login|logout|register|registrierung|anmelden|cookie|privacy|agb|faq|kontakt|contact|profil|account|newsletter|user:|benutzer:|hilfe:|help:|vorlage:|template:|spezial:|special:|discussion|diskussion)([/_?&=-]|$)', re.I)
NEG_TITLE = re.compile(r'^(datenschutz|impressum|kontakt|faq|cookies|agb|registrieren|links auf diese seite|benutzer:|vorlage:|hilfe:|spezial:)',re.I)
OVERVIEW = re.compile(r'(überblick|übersicht|startseite|inhaltsverzeichnis|alle materialien|themenblätter im unterricht|methoden- und spielekiste|materialien für mehr sicherheit im netz)',re.I)
PATH_OVERVIEW = {'/','/unterrichtseinheiten','/wiki/unterrichtsmethoden','/unterrichtsmethoden','/wiki/unterrichtsideen','/wiki/methode','/lernen/angebote','/shop/materialien/themenblaetter','/downloads','/materialien','/arbeitsblaetter','/unterrichtsbausteine','/unterrichtsvorschlaege'}
LINK_HINT = re.compile(r'unterricht|arbeitsblatt|material|methode|didakt|lernpfad|lernbaustein|lernmodul|verlauf|stundenentwurf|bildung|schule|oer|download|aufgabe|geschichte|informatik|physik|chemie|geographie|mathematik|politik|lesson|worksheet|wiki|uebung|übung',re.I)
TASK = re.compile(r'arbeitsauftr[aä]g|arbeitsanweis|aufgabenstellung|aufgabe\s*\d+|bearbeite|analysiere|vergleiche|diskutiere|untersuche|recherchiere|erarbeite|arbeitsblatt|lernauftrag|worksheet',re.I)
LESSON = re.compile(r'unterrichtsverlauf|verlaufsplan|stundenverlauf|stundenentwurf|unterrichtsphase|einstiegsphase|erarbeitungsphase|sicherungsphase|zeitbedarf|unterrichtseinheit|unterrichtsstunde|unterrichtsmodul|45\s*min|90\s*min',re.I)
METHOD = re.compile(r'methodische\s+(hinweise|anregungen|umsetzung)|methodenbeschreibung|sozialform|gruppenpuzzle|partnerarbeit|gruppenarbeit|stationenlernen|rollenspiel|fishbowl|methodik|methodisch.didaktisch',re.I)
DIDACTIC = re.compile(r'didaktische\s+(hinweise|anregungen|kommentar|handreichung)|lehrerhandreichung|handreichung\s+f[uü]r|unterrichtsvorschlag|lernziel|lernziele|kompetenzziel|differenzierung|sekundarstufe|unterrichtsmaterial|lehrkr[aä]fte',re.I)
DOCUMENT_TITLE = re.compile(r'unterricht|arbeitsblatt|lernpfad|methode|aufgabe|handreichung|stunde|unterrichtsvorschlag',re.I)
CC = re.compile(r'\b(?:CC[ -]?(?:BY(?:[ -](?:SA|NC(?:[ -]SA)?))?|0)|CC0|public domain)\b',re.I)


def utc(): return datetime.now(timezone.utc).isoformat()


def canon(url):
    try:
        u=html.unescape(url or '').strip()
        p=urlsplit(u)
        h=(p.hostname or '').lower()
        if p.scheme not in ('http','https') or not h or h not in ALLOWED_HOSTS or p.username or p.password or p.port not in (None,80,443) or len(u)>2048:
            return ''
        path=re.sub(r'/{2,}','/',p.path or '/')
        path=path.rstrip('/') or '/'
        query=urlencode([(k,v) for k,v in parse_qsl(p.query,keep_blank_values=True) if k.lower() not in TRACKING and not k.lower().startswith(('utm_','mtm_','pk_'))])
        return urlunsplit(('https',h,path,query,''))
    except Exception:return ''
canonical=canon


def relevant_path(u,seed=False):
    p=urlsplit(u); path=p.path.lower()
    if NEG_URL.search(path+'?'+p.query): return False
    if path.endswith(('.css','.js','.jpg','.jpeg','.png','.webp','.gif','.svg','.mp3','.mp4','.zip','.exe','.woff','.woff2','.ico')):return False
    if seed or path.endswith(SUFFIXES):return True
    if p.hostname in ('unterrichten.zum.de','deutsch-lernen.zum.de') and path.startswith('/wiki/'):return True
    return bool(LINK_HINT.search(path+' '+p.query))


def clean_text(text):
    lines=[re.sub(r'\s+',' ',x).strip() for x in (text or '').splitlines()]
    return '\n'.join(x for x in lines if x)[:400000]


def license_from_html(soup, u):
    if urlsplit(u).hostname in SITE_LICENSE:return SITE_LICENSE[urlsplit(u).hostname]
    for tag in soup.select('link[rel~=license],a[rel~=license],meta[name*=license],meta[name*=rights],meta[property*=license]'):
        value=tag.get('href') or tag.get('content') or tag.get_text(' ',strip=True)
        if value and ('creativecommons.org/licenses/' in value.lower() or CC.search(value)):
            return value[:250]
    return ''


def license_from_pdf_text(text):
    for match in CC.finditer(text):
        around=text[max(0,match.start()-150):match.end()+100].lower()
        if re.search(r'lizen[sz]iert|lizenz|creative commons|dieses werk|dieses material|steht unter|veröffentlicht unter|licensed under',around):
            return match.group().upper()
    return ''


def parse_html(raw,u):
    soup=BeautifulSoup(raw,'lxml')
    title=soup.title.get_text(' ',strip=True) if soup.title else ''
    h1=soup.find('h1')
    if h1 and (h1.get_text(' ',strip=True).lower() not in title.lower()):title=h1.get_text(' ',strip=True)
    lic=license_from_html(soup,u)
    main=None
    for css in ('.mw-parser-output','article .entry-content','article','main','[role=main]','#content','.content'):
        choice=soup.select_one(css)
        if choice and len(choice.get_text(' ',strip=True))>=140:
            main=choice;break
    main=main or soup.body or soup
    links=[]
    for a in main.select('a[href]'):
        link=canon(urljoin(u,a['href']))
        if link:links.append((link,a.get_text(' ',strip=True)[:130]))
    for node in main.select('script,style,noscript,header,footer,nav,aside,form,button,svg,iframe,.mw-editsection,.toc,.navbox,.catlinks,.printfooter,.breadcrumb,.breadcrumbs,.sidebar,.social-share,[role=navigation]'):
        node.decompose()
    headings=[x.get_text(' ',strip=True) for x in main.select('h1,h2,h3')][:55]
    canonical_tag=soup.select_one('link[rel=canonical]')
    declared=canon(urljoin(u,canonical_tag.get('href',''))) if canonical_tag else ''
    return {'title':title or urlsplit(u).path.split('/')[-1], 'text':clean_text(main.get_text('\n',strip=True)),
            'headings':headings,'links':list(dict.fromkeys(links)), 'license':lic,'canonical':declared or u}


def classify(title,u,text,headings,links):
    path=urlsplit(u).path.lower().rstrip('/') or '/'
    base=re.split(r'\s+[|–—]\s+',title,maxsplit=1)[0].strip().lower()
    if NEG_URL.search(path) or NEG_TITLE.search(title):return 'excluded',0,['negative_title_or_url'],False
    if path in PATH_OVERVIEW or OVERVIEW.search(base) or base in ('unterrichtsmethoden','unterrichtsideen','angebote'):
        return 'overview',0,['overview_navigation_only'],False
    blob=' '.join([title,*headings,text]);ev={
        'task':bool(TASK.search(blob)),
        'lesson':bool(LESSON.search(blob)) or bool(re.search(r'\bmodul\s*\d+\b',blob[:1200],re.I) and re.search(r'\bUE[- –]?[a-z]\b|material. und arbeitsblätter',blob,re.I)),
        'method':bool(METHOD.search(blob)), 'didactic':bool(DIDACTIC.search(blob)),
        'attachment':any(urlsplit(x).path.lower().endswith(SUFFIXES) for x,_ in links)}
    score=3*sum(ev[k] for k in ('task','lesson','method','didactic'))+int(ev['attachment'])+int(len(text)>1600)+int(bool(DOCUMENT_TITLE.search(title)))
    reasons=[k for k,v in ev.items() if v]
    accepted=len(text)>=MIN_TEXT_CHARS and sum(ev.values())>=2 and score>=6
    if not accepted:return ('review_candidate' if reasons else 'other'),score,reasons,False
    if ev['task'] and ev['lesson']:kind='lesson_plan'
    elif ev['method'] and ev['didactic']:kind='didactic_guide'
    elif ev['task'] and re.search(r'arbeitsblatt|worksheet',title,re.I):kind='worksheet'
    elif ev['method']:kind='teaching_method'
    elif ev['didactic'] and ev['lesson']:kind='didactic_guide'
    else:kind='teaching_activity'
    return kind,score,reasons,True


def parse_document(raw,ext):
    if ext=='.pdf':
        if not raw.startswith(b'%PDF'):raise ValueError('invalid pdf')
        from pypdf import PdfReader
        return clean_text('\n'.join(p.extract_text() or '' for p in PdfReader(io.BytesIO(raw),strict=False).pages[:250]))
    if ext=='.docx':
        from docx import Document
        doc=Document(io.BytesIO(raw))
        return clean_text('\n'.join([p.text for p in doc.paragraphs]+[' | '.join(c.text for c in r.cells) for t in doc.tables for r in t.rows]))
    if ext=='.pptx':
        from pptx import Presentation
        slides=Presentation(io.BytesIO(raw)).slides
        return clean_text('\n'.join(sh.text for sl in slides for sh in sl.shapes if sh.has_text_frame))
    if ext in ('.odt','.odp'):
        with zipfile.ZipFile(io.BytesIO(raw)) as z:root=ET.fromstring(z.read('content.xml'))
        return clean_text('\n'.join(' '.join(e.itertext()) for e in root.iter() if e.tag.endswith(('}p','}h'))))
    if ext=='.txt':return clean_text(raw.decode('utf-8-sig',errors='replace'))
    raise ValueError('unsupported document')


def ext_from(u,headers):
    path=urlsplit(u).path.lower()
    for x in SUFFIXES:
        if path.endswith(x):return x
    mime=headers.get('Content-Type','').lower()
    for m,x in [('application/pdf','.pdf'),('wordprocessingml','.docx'),('presentationml','.pptx'),('opendocument.text','.odt'),('text/plain','.txt')]:
        if m in mime:return x
    return ''

class TeachingCrawler:
    def __init__(self,out):
        self.dir=Path(out);self.dir.mkdir(parents=True,exist_ok=True)
        self.material_dir=self.dir/'materials';self.material_dir.mkdir(exist_ok=True)
        self.db=sqlite3.connect(self.dir/'crawler_state.sqlite',check_same_thread=False,timeout=40)
        self.db.execute('PRAGMA journal_mode=WAL');self.db.execute('PRAGMA busy_timeout=40000')
        self.db.execute("""CREATE TABLE IF NOT EXISTS frontier (
            url TEXT PRIMARY KEY,kind TEXT,depth INTEGER,priority INTEGER,parent TEXT,
            inherited_license TEXT,status TEXT DEFAULT 'queued',error TEXT)""")
        self.db.execute("CREATE INDEX IF NOT EXISTS frontier_pick ON frontier(status,priority DESC,depth)")
        self.db.execute("""CREATE TABLE IF NOT EXISTS documents (
            docno TEXT PRIMARY KEY,url TEXT UNIQUE,title TEXT,body TEXT,fingerprint TEXT UNIQUE,
            doc_type TEXT,source TEXT,license TEXT,parent TEXT,metadata TEXT)""")
        self.db.execute("""CREATE TABLE IF NOT EXISTS material_links (
            url TEXT PRIMARY KEY,parent TEXT,license TEXT,created TEXT)""")
        self.db.execute("""CREATE TABLE IF NOT EXISTS files (
            url TEXT PRIMARY KEY,path TEXT,size INTEGER,text_chars INTEGER,status TEXT,license TEXT,parent TEXT)""")
        self.db.execute("UPDATE frontier SET status='queued' WHERE status='running'")
        self.db.commit()
        self.lock=threading.RLock();self.stop=threading.Event()
        self.sessions=threading.local();self.host_locks=defaultdict(threading.Lock)
        self.host_last=defaultdict(float);self.robots={};self.host_count=Counter();self.stats=Counter()
        self.start_time=time.monotonic();self.last_progress=0
        self.seen_count=self.db.execute('SELECT COUNT(*) FROM frontier').fetchone()[0]
        self.processed=self.db.execute("SELECT COUNT(*) FROM frontier WHERE status IN ('done','error','blocked') AND kind!='sitemap'").fetchone()[0]
        for (u,) in self.db.execute("SELECT url FROM frontier WHERE status='done'"):
            self.host_count[urlsplit(u).hostname]+=1
        self.document_bytes=0
        for id_,u,t,body in self.db.execute('SELECT docno,url,title,body FROM documents'):
            self.document_bytes+=len((json.dumps({'docno':id_,'url':u,'title':t,'text':body},ensure_ascii=False)+'\n').encode('utf-8'))
        self.raw_file=(self.dir/'crawl.jsonl').open('a',encoding='utf-8')
        self.manifest_file=(self.dir/'material_manifest.jsonl').open('a',encoding='utf-8')
        self.log_file=(self.dir/'progress.log').open('a',encoding='utf-8')
        if not self.seen_count:self.seed_frontier()
        self.log(f'Bereit: {self.seen_count} URLs in der Warteschlange, {self.processed} verarbeitet, '
                 f'{self.db.execute("SELECT COUNT(*) FROM documents").fetchone()[0]} indexiert')

    def log(self,message):
        with self.lock:
            line=f'[{datetime.now().strftime("%H:%M:%S")}] {message}'
            print(line,flush=True);self.log_file.write(line+'\n');self.log_file.flush()

    def queue_url(self,u,depth,priority=0,kind=None,parent='',license_='',seed=False):
        u=canon(u)
        if not u or depth>PROFILE['max_depth']:return False
        if not kind:kind='material' if urlsplit(u).path.lower().endswith(SUFFIXES) else 'html'
        if kind=='material':
            if not DOWNLOAD_MATERIALS:return False
            pdf_check=CHECK_LICENSE_IN_PDFS and urlsplit(u).path.lower().endswith('.pdf')
            if ONLY_OPEN_LICENSE_MATERIALS and not (license_ or SITE_LICENSE.get(urlsplit(u).hostname) or pdf_check):return False
        elif kind!='sitemap' and not relevant_path(u,seed):return False
        with self.lock:
            if self.seen_count>=PROFILE['max_queue'] or self.stop.is_set():return False
            cur=self.db.execute('INSERT OR IGNORE INTO frontier (url,kind,depth,priority,parent,inherited_license,status) VALUES (?,?,?,?,?,?,\'queued\')',
                                (u,kind,depth,priority,parent,license_))
            if cur.rowcount:
                self.seen_count+=1;self.db.commit();return True
        return False

    def seed_frontier(self):
        for seed in SEEDS:self.queue_url(seed,0,priority=25,kind='html',seed=True)
        if SITEMAPS_AKTIV and MODUS!='TEST':
            for host in sorted(ALLOWED_HOSTS):self.queue_url('https://'+host+'/sitemap.xml',0,-30,kind='sitemap',seed=True)

    def session(self):
        if not hasattr(self.sessions,'http'):
            self.sessions.http=requests.Session()
            self.sessions.http.headers.update({'User-Agent':USER_AGENT})
        return self.sessions.http

    def robots_allow(self,u):
        host=urlsplit(u).hostname
        if host not in self.robots:
            try:
                r=self.session().get(f'https://{host}/robots.txt',timeout=(7,18),allow_redirects=False)
                if r.status_code in (404,410):self.robots[host]=(None,0)
                elif r.status_code==200:
                    rp=urllib.robotparser.RobotFileParser();rp.parse(r.text.splitlines())
                    self.robots[host]=(rp,rp.crawl_delay(USER_AGENT) or rp.crawl_delay('*') or 0)
                else:self.robots[host]=(False,0)
            except requests.RequestException:self.robots[host]=(False,0)
        rp,delay=self.robots[host]
        return (rp is None or (rp is not False and rp.can_fetch(USER_AGENT,u))),float(delay)

    def fetch(self,u,kind):
        for redirect in range(5):
            host=urlsplit(u).hostname
            if host not in ALLOWED_HOSTS:raise PermissionError('redirect_outside_curated_domains')
            with self.host_locks[host]:
                allowed,crawl_delay=self.robots_allow(u)
                if not allowed:raise PermissionError('robots_disallow_or_unavailable')
                delay=max(DOWNLOAD_DELAY_SECONDS,crawl_delay)
                remain=delay-(time.monotonic()-self.host_last[host])
                if remain>0:time.sleep(remain)
                self.host_last[host]=time.monotonic()
                response=self.session().get(u,timeout=(8,35),allow_redirects=False,stream=True)
                if response.status_code in (301,302,303,307,308):
                    target=canon(urljoin(u,response.headers.get('Location','')))
                    response.close()
                    if not target:raise PermissionError('unapproved_redirect')
                    u=target;continue
                if response.status_code!=200:
                    status=response.status_code;response.close();raise ValueError(f'HTTP_{status}')
                maxbytes=MAX_MATERIAL_MIB*1024**2 if kind=='material' else 4*1024**2
                if int(response.headers.get('Content-Length',0) or 0)>maxbytes:
                    response.close();raise ValueError('file_too_large')
                body=bytearray()
                try:
                    for part in response.iter_content(65536):
                        body.extend(part)
                        if len(body)>maxbytes:raise ValueError('response_too_large')
                    headers=dict(response.headers)
                finally:response.close()
                return u,bytes(body),headers
        raise ValueError('too_many_redirects')

    def claim(self):
        with self.lock:
            item=self.db.execute("SELECT url,kind,depth,priority,parent,inherited_license FROM frontier WHERE status='queued' ORDER BY priority DESC,depth LIMIT 1").fetchone()
            if item:
                self.db.execute("UPDATE frontier SET status='running' WHERE url=?",(item[0],));self.db.commit()
            return item

    def complete(self,u,status='done',error=''):
        with self.lock:
            self.db.execute('UPDATE frontier SET status=?,error=? WHERE url=?',(status,error[:250],u));self.db.commit()
            self.processed+=1

    def index(self,rec):
        body=rec['text'];digest=hashlib.sha256(re.sub(r'\s+',' ',body).encode('utf-8')).hexdigest()
        id_=hashlib.sha256(rec['url'].encode('utf-8')).hexdigest()
        meta={k:v for k,v in rec.items() if k not in ('url','title','text','doc_type','source','license','parent')}
        with self.lock:
            cur=self.db.execute('INSERT OR IGNORE INTO documents VALUES(?,?,?,?,?,?,?,?,?,?)',
                (id_,rec['url'],rec['title'],body,digest,rec['doc_type'],rec.get('source'),rec.get('license'),
                 rec.get('parent'),json.dumps(meta,ensure_ascii=False)))
            if cur.rowcount:
                self.db.commit();self.stats['indexed']+=1
                self.document_bytes+=len((json.dumps({'docno':id_,'url':rec['url'],'title':rec['title'],'text':body},ensure_ascii=False)+'\n').encode('utf-8'))
                if self.document_bytes>=TARGET_MIB*1024**2:self.stop.set()
                return True
            self.stats['duplicate_index']+=1;return False

    def handle_html(self,u,data,headers,depth):
        ct=headers.get('Content-Type','').lower()
        if 'html' not in ct and not data[:100].lstrip().lower().startswith((b'<html',b'<!doctype html')):
            self.stats['wrong_content_type']+=1;return
        enc=re.search(r'charset=([\w-]+)',ct)
        parsed=parse_html(data.decode(enc.group(1) if enc else 'utf-8',errors='replace'),u)
        title=parsed['title'];text=parsed['text'];links=parsed['links'];lic=parsed['license']
        category,score,signals,indexed=classify(title,u,text,parsed['headings'],links)
        attachments=[link for link,_ in links if urlsplit(link).path.lower().endswith(SUFFIXES)]
        with self.lock:
            self.raw_file.write(json.dumps({'url':u,'title':title,'page_type':category,'quality_score':score,
                'indexed':indexed,'text':text[:20000],'full_text_length':len(text),
                'material_links':attachments,'license':lic,'crawled_at':utc()},ensure_ascii=False)+'\n')
            self.raw_file.flush()
            for link in attachments:
                self.db.execute('INSERT OR IGNORE INTO material_links VALUES (?,?,?,?)',(link,u,lic,utc()))
            self.db.commit()
        self.stats['html_'+category]+=1
        if indexed:
            self.index({'url':parsed['canonical'],'title':title,'text':text,'doc_type':category,
                        'source':HOST_TO_SOURCE.get(urlsplit(u).hostname),'license':lic,'parent':'',
                        'quality_score':score,'evidence':signals,'material_links':attachments,
                        'headings':parsed['headings'],'crawled_at':utc()})
        if depth<PROFILE['max_depth']:
            links.sort(key=lambda pair: bool(LINK_HINT.search(pair[0]+' '+pair[1])),reverse=True)
            for target,anchor in links[:250]:
                if not QUELLEN_UEBERGREIFEND and urlsplit(target).hostname!=urlsplit(u).hostname:continue
                isfile=urlsplit(target).path.lower().endswith(SUFFIXES)
                if isfile:
                    self.stats['material_links_seen']+=1
                    self.queue_url(target,depth+1,70,kind='material',parent=u,license_=lic)
                else:
                    self.queue_url(target,depth+1,10 if LINK_HINT.search(target+' '+anchor) else -8,parent=u)

    def handle_material(self,u,raw,headers,parent,license_):
        ext=ext_from(u,headers)
        if ext not in SUFFIXES:return
        if not license_:license_=SITE_LICENSE.get(urlsplit(u).hostname,'')
        if ONLY_OPEN_LICENSE_MATERIALS and not license_ and not (CHECK_LICENSE_IN_PDFS and ext=='.pdf'):
            self.stats['not_open_licensed']+=1;return
        try:body=parse_document(raw,ext)
        except Exception as exc:
            self.stats['material_parse_failed']+=1
            body=''
        if not license_ and CHECK_LICENSE_IN_PDFS and ext=='.pdf':license_=license_from_pdf_text(body)
        if ONLY_OPEN_LICENSE_MATERIALS and not license_:
            self.stats['not_open_licensed']+=1;return
        digest=hashlib.sha256(raw).hexdigest()
        dest=self.material_dir/(digest+ext)
        if not dest.exists():dest.write_bytes(raw)
        status='saved_no_search_text'
        if len(body)>=max(100,MIN_TEXT_CHARS//2):
            title=Path(urlsplit(u).path).name
            kind,score,signals,accepted=classify(title,u,body,[],[])
            if accepted or TASK.search(body) or LESSON.search(body) or METHOD.search(body) or DIDACTIC.search(body) or DOCUMENT_TITLE.search(title):
                if not accepted:kind='worksheet' if TASK.search(body) else 'didactic_material'
                status='indexed' if self.index({'url':u,'title':title,'text':body,'doc_type':kind,
                    'source':HOST_TO_SOURCE.get(urlsplit(u).hostname),'license':license_,
                    'parent':parent,'file_path':str(dest),'sha256':digest,'crawled_at':utc()}) else 'duplicate'
        with self.lock:
            self.db.execute('INSERT OR REPLACE INTO files VALUES(?,?,?,?,?,?,?)',(u,str(dest),len(raw),len(body),status,license_,parent))
            self.db.commit()
            self.manifest_file.write(json.dumps({'url':u,'parent_url':parent,'path':str(dest),
                'license':license_,'bytes':len(raw),'text_chars':len(body),'sha256':digest,'status':status},ensure_ascii=False)+'\n')
            self.manifest_file.flush()
        self.stats['material_saved']+=1

    def handle_sitemap(self,u,raw,depth):
        try:
            if raw.startswith(b'\x1f\x8b'):raw=gzip.decompress(raw)
            root=ET.fromstring(raw)
        except Exception:self.stats['invalid_sitemap']+=1;return
        n=0
        for child in root.iter():
            if child.tag.endswith('}loc') or child.tag=='loc':
                loc=canon((child.text or '').strip())
                if not loc:continue
                n+=1
                if n>1800:break
                if loc.lower().endswith(('.xml','.xml.gz')):
                    if depth<2:self.queue_url(loc,depth+1,-30,kind='sitemap',seed=True)
                else:self.queue_url(loc,0,-10,seed=False)
        self.stats['sitemap_links']+=n
    def worker(self):
        while not self.stop.is_set():
            if time.monotonic()-self.start_time>PROFILE['runtime_minutes']*60 or self.processed>=PROFILE['max_pages']:
                self.stop.set();break
            item=self.claim()
            if not item:
                with self.lock:
                    active=self.db.execute("SELECT COUNT(*) FROM frontier WHERE status='running'").fetchone()[0]
                if not active:break
                time.sleep(.3);continue
            u,kind,depth,priority,parent,license_=item;host=urlsplit(u).hostname
            try:
                if self.host_count[host]>=PROFILE['max_per_host']:
                    self.complete(u,'blocked','max_host_pages');continue
                if kind=='material' and ONLY_OPEN_LICENSE_MATERIALS and not (license_ or SITE_LICENSE.get(host) or (CHECK_LICENSE_IN_PDFS and u.lower().endswith('.pdf'))):
                    self.complete(u,'blocked','no_licence');continue
                actual,data,headers=self.fetch(u,kind)
                if kind=='sitemap':self.handle_sitemap(actual,data,depth)
                elif kind=='material':self.handle_material(actual,data,headers,parent,license_)
                else:self.handle_html(actual,data,headers,depth)
                self.host_count[host]+=1
                self.complete(u)
            except PermissionError as exc:
                self.stats['robots_or_redirect_blocked']+=1
                self.complete(u,'blocked',str(exc))
            except Exception as exc:
                self.stats['fetch_or_parse_error']+=1
                self.complete(u,'error',str(exc))
            if self.processed%25==0 or time.monotonic()-self.last_progress>60:
                with self.lock:
                    if time.monotonic()-self.last_progress>2:
                        self.last_progress=time.monotonic()
                        docs=self.db.execute('SELECT COUNT(*) FROM documents').fetchone()[0]
                        self.log(f'Fortschritt: {self.processed} Abrufe, {docs} Dokumente, '
                                 f'{self.document_bytes/1024**2:.2f}/{TARGET_MIB} MiB, '
                                 f'{self.stats["material_saved"]} gespeicherte Materialdateien')

    def crawl(self):
        self.log(f'START {MODUS}: {len(SEEDS)} Seeds, {PROFILE["workers"]} parallele Worker, Materialdownload={DOWNLOAD_MATERIALS}')
        with concurrent.futures.ThreadPoolExecutor(max_workers=PROFILE['workers']) as pool:
            for f in [pool.submit(self.worker) for _ in range(PROFILE['workers'])]:f.result()
        self.log(f'ENDE: {self.processed} Seiten, {self.document_bytes/1024**2:.2f} MiB bereinigte Daten')

    def close(self):
        with self.lock:
            for f in (self.raw_file,self.manifest_file,self.log_file):
                try:f.flush();f.close()
                except Exception:pass
            self.db.commit();self.db.close()


def export_reports(db_path,out,target):
    out=Path(out);db=sqlite3.connect(db_path)
    counts=Counter();sources=Counter();text_bytes=0;n=0
    with (out/'documents.jsonl').open('w',encoding='utf-8') as course, (out/'search_documents.jsonl').open('w',encoding='utf-8') as details:
        for id_,u,t,body,fp,kind,src,lic,parent,meta in db.execute('SELECT * FROM documents ORDER BY docno'):
            course.write(json.dumps({'docno':id_,'url':u,'title':t,'text':body},ensure_ascii=False)+'\n')
            full={'docno':id_,'url':u,'title':t,'search_text':body,'content_fingerprint':fp,
                  'document_type':kind,'source':src,'license':lic,'parent_url':parent}
            full.update(json.loads(meta or '{}'))
            details.write(json.dumps(full,ensure_ascii=False)+'\n')
            counts[kind]+=1;sources[src or 'unbekannt']+=1;text_bytes+=len(body.encode('utf-8'));n+=1
    with (out/'material_links_discovered.jsonl').open('w',encoding='utf-8') as material_list:
        for u,parent,lic,created in db.execute('SELECT * FROM material_links ORDER BY url'):
            material_list.write(json.dumps({'url':u,'parent_url':parent,'source_license':lic,'discovered_at':created},ensure_ascii=False)+'\n')
    sizes=lambda p: p.stat().st_size if p.exists() else 0
    bodybytes=sizes(out/'documents.jsonl')
    mat_bytes=db.execute('SELECT COALESCE(SUM(size),0) FROM files').fetchone()[0]
    mats=dict(db.execute('SELECT status,COUNT(*) FROM files GROUP BY status').fetchall())
    tasks=dict(db.execute('SELECT status,COUNT(*) FROM frontier GROUP BY status').fetchall())
    report={'indexed_documents':n,'document_types':dict(counts),'sources':dict(sources),
            'downloaded_files':mats,'discovered_material_links':db.execute('SELECT COUNT(*) FROM material_links').fetchone()[0],
            'crawl_status':tasks,'report_generated_utc':utc()}
    size_report={'target_mib':target,'documents_mib':round(bodybytes/1024**2,2),
                 'documents_bytes':bodybytes,'search_text_bytes':text_bytes,
                 'search_documents_mib':round(sizes(out/'search_documents.jsonl')/1024**2,2),
                 'raw_crawl_mib':round(sizes(out/'crawl.jsonl')/1024**2,2),
                 'material_file_bytes':mat_bytes,'target_reached':bodybytes>=target*1024**2}
    (out/'quality_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    (out/'corpus_size_report.json').write_text(json.dumps(size_report,ensure_ascii=False,indent=2),encoding='utf-8')
    (out/'run_info.json').write_text(json.dumps({'run_name':RUN_NAME,'mode':MODUS,'profile':PROFILE,
        'seeds':SEEDS,'allowed_hosts':sorted(ALLOWED_HOSTS),'download_files':DOWNLOAD_MATERIALS,
        'only_open_licensed_files':ONLY_OPEN_LICENSE_MATERIALS},ensure_ascii=False,indent=2),encoding='utf-8')
    db.close()
    print('\nEXPORT FERTIG:',n,'indexierte Dokumente;',size_report['documents_mib'],'MiB;',
          len(mats),'Materialstatus-Gruppen',flush=True)
    print('Korpus:',(out/'documents.jsonl').resolve(),flush=True)
    print('Materialdateien:',(out/'materials').resolve(),flush=True)
    print('Ziel erreicht:',size_report['target_reached'],flush=True)
    return report,size_report


## 4. Offline-Selbsttest (vor dem Netzlauf)
Diese kleinen Regressionstests prüfen Materialerkennung, Navigationsausschluss und URL-Deduplizierung, ohne Internet zu benötigen. Ein Fehler verhindert den längeren Crawl.

In [ ]:
# Der Test arbeitet ohne Internet und legt keine Crawl-Ausgabe an.
unit = """<!doctype html><html><head><title>Unterricht: Quellenauswertung</title></head>
<body><main><h1>Unterrichtseinheit: Quellen auswerten</h1><h2>Unterrichtsverlauf</h2>
<p>Arbeitsauftrag: Analysiere die Quellen und vergleiche die Ergebnisse in Gruppenarbeit.</p>
<h2>Methodische Hinweise</h2><p>Lernziel und Einstiegsphase: Die Schülerinnen und Schüler
erschließen historisches Quellenmaterial.</p>""" + "<p>Die Arbeitsblätter werden in Partnerarbeit verglichen und die Ergebnisse im Plenum diskutiert.</p>"*8 + """
<a href="https://segu-geschichte.de/files/quellenanalyse.pdf">Arbeitsblatt als PDF</a></main></body></html>"""
parsed=parse_html(unit,'https://segu-geschichte.de/unterrichtsmaterial/quellenanalyse')
result=classify(parsed['title'],'https://segu-geschichte.de/unterrichtsmaterial/quellenanalyse',
                parsed['text'],parsed['headings'],parsed['links'])
assert result[-1] is True, ('Unterrichtseinheit nicht erkannt',result)
assert any(link.endswith('.pdf') for link,_ in parsed['links']), 'Materiallink nicht erkannt'
bad=classify('Datenschutz – ZUM-Unterrichten','https://unterrichten.zum.de/wiki/Datenschutz',
             'Arbeitsblatt Unterrichtsverlauf für den Unterricht '+('Lorem ipsum '*150),[],[])
assert bad[-1] is False, 'Datenschutz-Fehlklassifikation'
overview=classify('Unterrichtsmethoden – ZUM Deutsch Lernen','https://deutsch-lernen.zum.de/wiki/Unterrichtsmethoden',
             'Methoden und Lernziele in Gruppenarbeit '+('Lorem ipsum '*150),[],[])
assert overview[-1] is False, 'Methodenübersicht sollte nicht im Suchindex landen'
u1=canonical('https://segu-geschichte.de/arbeitsblaetter/?utm_source=test#top')
u2=canonical('https://segu-geschichte.de/arbeitsblaetter/')
assert u1==u2, ('URL-Deduplizierung falsch',u1,u2)
print('Offline-Tests bestanden: Unterrichtseinheit, Materiallink, Datenschutz, Navigation, URL-Normalisierung.',flush=True)

## 5. Crawl automatisch ausführen
**Diese Zelle startet bei Run All automatisch.** Du siehst alle 25 Abrufe oder mindestens alle 60 Sekunden einen Fortschrittsbericht. Der Zustand wird dauerhaft unter `outputs/runs/<RUN_NAME>/crawler_state.sqlite` gespeichert.  
Abbruch: „Interrupt Kernel“. Danach später wieder **Run All** mit demselben `RUN_NAME` starten.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
OUTPUT_DIR = Path("outputs") / "runs" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Ausgabeverzeichnis:", OUTPUT_DIR.resolve(), flush=True)
if START_CRAWL:
    runner = TeachingCrawler(OUTPUT_DIR)
    try:
        runner.crawl()
    except KeyboardInterrupt:
        print("Crawl unterbrochen. SQLite-Zustand bleibt erhalten; Export startet mit bisherigen Dokumenten.",flush=True)
    except Exception as exc:
        print("Crawl-Fehler:",repr(exc),"– Export der bisherigen Daten startet trotzdem.",flush=True)
    finally:
        runner.close()
    print("Netzlauf abgeschlossen oder beendet. Export startet.", flush=True)
else:
    print("START_CRAWL=False: Netzlauf ausdrücklich übersprungen.", flush=True)

## 6. Exporte und Qualitätsbericht erstellen
Der Export verwendet die **SQLite-Dokumente als Quelle**, so entstehen beim Wiederaufnehmen keine doppelten JSONL-Zeilen. Auch wenn der Crawl sein Zeit- oder Seitenlimit erreicht hat, wird alles bisher Gesammelte ausgegeben.

In [ ]:
import sqlite3, json, os
sqlite_path = OUTPUT_DIR / "crawler_state.sqlite"
if sqlite_path.exists():
    export_reports(sqlite_path, OUTPUT_DIR, TARGET_MIB)
else:
    print("Noch keine Datenbank vorhanden. Aktiviere START_CRAWL für einen Lauf.")

## 7. Beispieltreffer anzeigen
Zeigt die tatsächlich gespeicherten Unterrichtsdokumente mit Typ, Lizenz und URL. Kontrolliere die Stichprobe, bevor du den resultierenden Korpus weitergibst.

In [ ]:
if sqlite_path.exists():
    with sqlite3.connect(sqlite_path) as c:
        rows = c.execute("SELECT doc_type, title, url, substr(text,1,140) FROM documents ORDER BY RANDOM() LIMIT 12").fetchall()
    print(f"STICHPROBE ({len(rows)} Dokumente)")
    for kind,title,url,excerpt in rows:
        print(f"\n[{kind}] {title}\n{url}\n{excerpt} ...")

## Wo liegen deine Dateien?

| Datei/Ordner | Inhalt |
|---|---|
| `outputs/runs/<RUN_NAME>/materials/` | gespeicherte, textlich geprüfte und freigegebene PDF-/Office-Dateien |
| `outputs/runs/<RUN_NAME>/material_manifest.jsonl` | Tatsächlich gespeicherte Materialdateien, Hashes und Lizenzverweise |
| `outputs/runs/<RUN_NAME>/material_links_discovered.jsonl` | Alle gefundenen Materiallinks, auch ohne Freigabe |
| `outputs/runs/<RUN_NAME>/documents.jsonl` | **abgabefertiges Format**: `docno`, `url`, `title`, `text` |
| `outputs/runs/<RUN_NAME>/search_documents.jsonl` | umfangreichere Suchdaten mit Metadaten und Materialbeziehungen |
| `outputs/runs/<RUN_NAME>/crawl.jsonl` | technische Crawl-Ergebnisse inklusive ausgeschlossener Seiten |
| `outputs/runs/<RUN_NAME>/quality_report.json` | tatsächlich indexierte Dokumenttypen und Quellen |
| `outputs/runs/<RUN_NAME>/corpus_size_report.json` | Größe von Rohdaten, indexiertem Korpus, Text und Materialdateien |
| `outputs/runs/<RUN_NAME>/crawler_state.sqlite` | Wiederaufnahmezustand und deduplizierter Datenbestand |

**Wichtig:** 300 MiB bereinigte Bildungsinhalte lassen sich nicht durch einen Lauf garantieren. Wenn der Lauf weniger ergibt, steht die tatsächliche Größe im Bericht. Es werden keine Seiten künstlich vervielfältigt.